# GPU Budget Negotiation Arena - Colab Training Notebook

This notebook is the judge-facing path for setup, artifact generation, optional SFT/GRPO training, and pushing outputs back to GitHub or Hugging Face.


## 1. Mount Google Drive and clone

Use Drive for generated artifacts and checkpoints. Use GitHub for source code.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
os.environ['DRIVE_OUT'] = '/content/drive/MyDrive/gpu_budget_negotiation_arena'
os.makedirs(os.environ['DRIVE_OUT'], exist_ok=True)


In [ ]:
%cd /content
REPO_URL = "https://github.com/abhinavgautam01/GPU_Budget_Negotiation_Arena.git"
REPO_DIR = "/content/GPU_Budget_Negotiation_Arena"
BRANCH = "main"

import os
import subprocess

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    status = subprocess.run(["git", "-C", REPO_DIR, "status", "--porcelain"], check=True, capture_output=True, text=True).stdout.strip()
    if status:
        subprocess.run(["git", "-C", REPO_DIR, "stash", "push", "-u", "-m", "colab-autostash-before-sync"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", BRANCH], check=True)

%cd /content/GPU_Budget_Negotiation_Arena
!git log -1 --oneline
!python -m pip install -q -e ".[dev]"
!python -m pytest -q


## 2. Generate required local artifacts

This produces baseline eval, reward-loop comparison, SFT traces, SFT chat messages, plots, and a demo transcript.


In [ ]:
%cd /content/GPU_Budget_Negotiation_Arena
!python scripts/check_submission.py
!mkdir -p "$DRIVE_OUT"
!cp -r artifacts plots data "$DRIVE_OUT"/
!ls -lh "$DRIVE_OUT"/artifacts


## 3. Optional Hugging Face login

Use this only if you want to push a model checkpoint or dataset from Colab. Do not paste tokens into committed files.


In [ ]:
# Uncomment when needed.
# !python -m pip install -q huggingface_hub
# from huggingface_hub import notebook_login
# notebook_login()


## 4. Optional SFT warm start with Unsloth

Run this on a GPU runtime. The SFT stage teaches JSON action formatting and basic negotiation behavior before GRPO.


In [ ]:
# Uncomment on a GPU runtime.
# !python -m pip install -q unsloth trl transformers datasets accelerate peft
# from datasets import load_dataset
# from trl import SFTConfig, SFTTrainer
# from unsloth import FastLanguageModel
#
# MODEL_NAME = 'unsloth/Llama-3.2-3B-Instruct'
# SFT_OUTPUT = 'artifacts/sft-checkpoint'
# dataset = load_dataset('json', data_files='data/sft_messages.jsonl', split='train')
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name=MODEL_NAME,
#     max_seq_length=4096,
#     load_in_4bit=True,
# )
# model = FastLanguageModel.get_peft_model(model, r=16, lora_alpha=16, target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
# trainer = SFTTrainer(
#     model=model,
#     tokenizer=tokenizer,
#     train_dataset=dataset,
#     args=SFTConfig(output_dir=SFT_OUTPUT, max_steps=120, per_device_train_batch_size=1, gradient_accumulation_steps=4, learning_rate=2e-4),
# )
# trainer.train()
# trainer.save_model(SFT_OUTPUT)
# !cp -r artifacts/sft-checkpoint "$DRIVE_OUT"/


## 5. GRPO reward-loop handoff

The environment reward is already exposed through `env.step(action).reward`. Use the JSON parser below as a starting point for TRL reward wiring.


In [ ]:
import json
from gpu_budget_arena.env import GpuBudgetNegotiationEnv
from gpu_budget_arena.models import GpuNegotiationAction, ResetConfig

def json_format_reward(text: str) -> float:
    try:
        GpuNegotiationAction.model_validate_json(text)
        return 0.2
    except Exception:
        return -0.2

def one_step_env_reward(action_text: str, task_type: str = 'market_round', seed: int = 0) -> float:
    env = GpuBudgetNegotiationEnv()
    env.reset(ResetConfig(task_type=task_type, seed=seed))
    try:
        action = GpuNegotiationAction.model_validate_json(action_text)
    except Exception:
        return -0.2
    obs = env.step(action)
    return obs.reward

print(json_format_reward('{"action_type":"wait"}'), one_step_env_reward('{"action_type":"wait"}'))


## 6. Final submission handoff

Verify final artifacts, copy them to Drive, and optionally push regenerated compact artifacts to GitHub. Keep large checkpoints in Drive or a Hugging Face model repo.


In [ ]:
%cd /content/GPU_Budget_Negotiation_Arena
!ls -lh artifacts/judged_transcript.md artifacts/before_after_training.md artifacts/training_eval.json artifacts/training_curve.json artifacts/training_report.md plots/baseline_rewards.svg plots/reward_progress.json
!mkdir -p "$DRIVE_OUT"
!cp -r artifacts plots data "$DRIVE_OUT"/
!ls -lh "$DRIVE_OUT"/artifacts
!ls -lh "$DRIVE_OUT"/plots
!git status --short

# Optional: push regenerated compact artifacts back to GitHub from Colab.
# !git add artifacts/baseline_eval.json artifacts/demo_transcript.md artifacts/judged_transcript.md artifacts/before_after_training.md artifacts/training_eval.json artifacts/training_curve.json artifacts/training_report.md plots/baseline_rewards.svg plots/reward_progress.json data/sft_traces.jsonl data/sft_messages.jsonl
# !git commit -m "Regenerate final submission artifacts"
# !git push origin main


## 7. Deployment checks

After pushing to Hugging Face Spaces from your local machine, run these checks. If the Space is still building, wait a minute and rerun.


In [ ]:
!curl -sS https://abhinavgautam01-gpu-budget-negotiation-arena.hf.space/health || true
!curl -sS https://abhinavgautam01-gpu-budget-negotiation-arena.hf.space/tasks || true
